In [1]:
import pandas as pd
import numpy as np

# ── 1. Load chemtastes_cleaned_full.csv ──
chem = pd.read_csv("chemtastes_cleaned_full.csv")
print("Shape:", chem.shape)
print("Columns:", chem.columns.tolist())
print("\nClass taste unique values:")
print(chem["Class taste"].unique())
print("\nValue counts:")
print(chem["Class taste"].value_counts())

Shape: (4075, 26)
Columns: ['ID', 'Name', 'PubChem CID', 'CAS number', 'canonical SMILES', 'Taste', 'Class taste', 'Reference_(cod)/[pp]', 'taste_word_count', 'taste_has_multiple', 'smiles_length', 'num_carbons', 'num_oxygens', 'num_nitrogens', 'num_sulfurs', 'num_halogens', 'num_rings', 'num_aromatic', 'num_double_bonds', 'num_triple_bonds', 'num_brackets', 'num_branches', 'C_O_ratio', 'N_C_ratio', 'ring_aromatic_ratio', 'heteroatom_ratio']

Class taste unique values:
['Sweetness' 'Bitterness' 'Umaminess' 'Sourness' 'Saltiness' 'Multitaste'
 'Tastelessness' 'Non-sweetness' 'Non-bitterness' 'Miscellaneous']

Value counts:
Class taste
Bitterness        1615
Sweetness         1313
Non-sweetness      304
Tastelessness      232
Umaminess          220
Multitaste         179
Miscellaneous      119
Sourness            49
Non-bitterness      28
Saltiness           16
Name: count, dtype: int64


In [2]:
# ── 2. Keep only canonical SMILES & Class taste ──
chem = chem[["canonical SMILES", "Class taste"]].copy()

# ── 3. Define the mapping ──
# Clear mappings:
#   Sweetness      -> sweet
#   Bitterness     -> bitter
#   Umaminess      -> umami
#   Sourness       -> sour
#   Saltiness      -> undefined   (no direct target)
#   Tastelessness  -> undefined
#   Non-sweetness  -> undefined   (negation – not a clear positive taste)
#   Non-bitterness -> undefined   (negation – not a clear positive taste)
#   Miscellaneous  -> undefined
#
# Ambiguous (REMOVE):
#   Multitaste     -> ambiguous, maps to multiple tastes → drop

taste_map = {
    "Sweetness":      "sweet",
    "Bitterness":     "bitter",
    "Umaminess":      "umami",
    "Sourness":       "sour",
    "Saltiness":      "undefined",
    "Tastelessness":  "undefined",
    "Non-sweetness":  "undefined",
    "Non-bitterness": "undefined",
    "Miscellaneous":  "undefined",
}

# Remove ambiguous rows (Multitaste)
ambiguous = ["Multitaste"]
before = len(chem)
chem = chem[~chem["Class taste"].isin(ambiguous)].copy()
print(f"Removed {before - len(chem)} ambiguous (Multitaste) rows  →  {len(chem)} remaining")

# Apply mapping
chem["Canonicalized Taste"] = chem["Class taste"].map(taste_map)

# Safety check – no unmapped values
assert chem["Canonicalized Taste"].isna().sum() == 0, "Unmapped values found!"

print("\nMapped value counts:")
print(chem["Canonicalized Taste"].value_counts())
chem.head()

Removed 179 ambiguous (Multitaste) rows  →  3896 remaining

Mapped value counts:
Canonicalized Taste
bitter       1615
sweet        1313
undefined     699
umami         220
sour           49
Name: count, dtype: int64


,canonical SMILES,Class taste,Canonicalized Taste
0,Oc1cc2c(cc1O)C1c3ccc(c(c3OCC1(O)C2)O)O,Sweetness,sweet
1,CC(C)=CCCC(C)(O)C1CC(O)C(=CC1=O)C,Sweetness,sweet
2,CC(=O)OC1C(Oc2cc(cc(c2C1=O)O)O)c1ccc(c(c1)O)O,Sweetness,sweet
3,COc1c(cc2c(c1O)C(=O)C(OC(C)=O)C(O2)c1ccc(c(c1)...,Sweetness,sweet
4,Oc1cc2c(cc1O)C1c3ccc(c(c3OCC1(O)C2)O)O,Sweetness,sweet


In [3]:
# ── 4. Load filtered_db.csv ──
db = pd.read_csv("filtered_db.csv")
print("filtered_db shape:", db.shape)
print("Columns:", db.columns.tolist())
print("\nSource value counts:")
print(db["Source"].value_counts())
print("\nTaste value counts:")
print(db["Canonicalized Taste"].value_counts())

filtered_db shape: (10374, 4)
Columns: ['Canonicalized SMILES', 'Standardized SMILES', 'Canonicalized Taste', 'Source']

Source value counts:
Source
flavor_db                      6794
chemtastes_db                  1447
IUPAC Dissocation Constants    1408
phytocompounds_db               693
tas2r_agonists                   32
Name: count, dtype: int64

Taste value counts:
Canonicalized Taste
sweet        5483
undefined    2010
sour         1489
bitter       1362
umami          30
Name: count, dtype: int64


In [4]:
# ── 5. Remove old chemtastes_db rows from filtered_db ──
db_no_chem = db[db["Source"] != "chemtastes_db"].copy()
print(f"Removed {len(db) - len(db_no_chem)} old chemtastes_db rows")
print(f"Remaining: {len(db_no_chem)} rows")

# ── 6. Prepare new chemtastes rows (match filtered_db schema) ──
# filtered_db columns: Canonicalized SMILES, Standardized SMILES, Canonicalized Taste, Source
# For chemtastes data, Standardized SMILES = canonical SMILES (same source)
chem_new = pd.DataFrame({
    "Canonicalized SMILES":  chem["canonical SMILES"].values,
    "Standardized SMILES":   chem["canonical SMILES"].values,
    "Canonicalized Taste":   chem["Canonicalized Taste"].values,
    "Source":                "chemtastes_db",
})
print(f"\nNew chemtastes_db rows to add: {len(chem_new)}")
print(chem_new["Canonicalized Taste"].value_counts())

# ── 7. Concatenate and reset index ──
db_updated = pd.concat([db_no_chem, chem_new], ignore_index=True)
print(f"\n=== Updated filtered_db shape: {db_updated.shape} ===")
print("\nSource value counts:")
print(db_updated["Source"].value_counts())
print("\nTaste value counts:")
print(db_updated["Canonicalized Taste"].value_counts())

Removed 1447 old chemtastes_db rows
Remaining: 8927 rows

New chemtastes_db rows to add: 3896
Canonicalized Taste
bitter       1615
sweet        1313
undefined     699
umami         220
sour           49
Name: count, dtype: int64

=== Updated filtered_db shape: (12823, 4) ===

Source value counts:
Source
flavor_db                      6794
chemtastes_db                  3896
IUPAC Dissocation Constants    1408
phytocompounds_db               693
tas2r_agonists                   32
Name: count, dtype: int64

Taste value counts:
Canonicalized Taste
sweet        6473
undefined    2398
bitter       2210
sour         1522
umami         220
Name: count, dtype: int64


In [5]:
# ── 8. Consistency checks before saving ──
# Ensure no nulls in any column
assert db_updated.isna().sum().sum() == 0, "Null values found!"

# Ensure taste values are only the expected set
expected_tastes = {"sweet", "bitter", "sour", "umami", "undefined"}
actual_tastes = set(db_updated["Canonicalized Taste"].unique())
assert actual_tastes == expected_tastes, f"Unexpected tastes: {actual_tastes - expected_tastes}"

# Ensure column schema matches original
assert list(db_updated.columns) == ["Canonicalized SMILES", "Standardized SMILES", "Canonicalized Taste", "Source"]

print("✅ All consistency checks passed!")
print(f"   Rows: {len(db_updated)}")
print(f"   Tastes: {sorted(actual_tastes)}")
print(f"   Sources: {sorted(db_updated['Source'].unique())}")

# ── 9. Save ──
db_updated.to_csv("filtered_db.csv", index=False)
print("\n💾 Saved to filtered_db.csv")

# Quick verify by re-reading
verify = pd.read_csv("filtered_db.csv")
print(f"   Verified: {verify.shape[0]} rows, {verify.shape[1]} columns")
verify.head()

✅ All consistency checks passed!
   Rows: 12823
   Tastes: ['bitter', 'sour', 'sweet', 'umami', 'undefined']
   Sources: ['IUPAC Dissocation Constants', 'chemtastes_db', 'flavor_db', 'phytocompounds_db', 'tas2r_agonists']

💾 Saved to filtered_db.csv
   Verified: 12823 rows, 4 columns


,Canonicalized SMILES,Standardized SMILES,Canonicalized Taste,Source
0,CC(O)CN,CC(O)CN,undefined,flavor_db
1,OCC1OC(O)CC(O)C1O,OCC1OC(O)CC(O)C1O,sweet,flavor_db
2,CC(C)C(=O)C(=O)O,CC(C)C(=O)C(=O)O,undefined,flavor_db
3,O=C(O)CCC(=O)C(=O)O,O=C(O)CCC(=O)C(=O)O,undefined,flavor_db
4,CCC(=O)C(=O)O,CCC(=O)C(=O)O,sweet,flavor_db


In [1]:
from datasets import load_dataset

ds = load_dataset("FartLabs/FartDB")
ds

c:\Users\Parv\OneDrive\Desktop\IP\Multi-Taste Predictor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    full: Dataset({
        features: ['Canonicalized SMILES', 'Standardized SMILES', 'Canonicalized Taste', 'Source'],
        num_rows: 15031
    })
    train: Dataset({
        features: ['Canonicalized SMILES', 'Standardized SMILES', 'Canonicalized Taste', 'Source'],
        num_rows: 10521
    })
    validation: Dataset({
        features: ['Canonicalized SMILES', 'Standardized SMILES', 'Canonicalized Taste', 'Source'],
        num_rows: 2255
    })
    test: Dataset({
        features: ['Canonicalized SMILES', 'Standardized SMILES', 'Canonicalized Taste', 'Source'],
        num_rows: 2255
    })
})

In [1]:
from datasets import load_dataset

ds = load_dataset("FartLabs/FartDB")

c:\Users\Parv\OneDrive\Desktop\IP\Multi-Taste Predictor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(ds)

DatasetDict({
    full: Dataset({
        features: ['Canonicalized SMILES', 'Standardized SMILES', 'Canonicalized Taste', 'Source'],
        num_rows: 15031
    })
    train: Dataset({
        features: ['Canonicalized SMILES', 'Standardized SMILES', 'Canonicalized Taste', 'Source'],
        num_rows: 10521
    })
    validation: Dataset({
        features: ['Canonicalized SMILES', 'Standardized SMILES', 'Canonicalized Taste', 'Source'],
        num_rows: 2255
    })
    test: Dataset({
        features: ['Canonicalized SMILES', 'Standardized SMILES', 'Canonicalized Taste', 'Source'],
        num_rows: 2255
    })
})


In [3]:
db=ds["full"].to_pandas()

In [4]:
db.columns

Index(['Canonicalized SMILES', 'Standardized SMILES', 'Canonicalized Taste',
       'Source'],
      dtype='object')

In [6]:
db["Canonicalized Taste"].value_counts()

Canonicalized Taste
sweet        9542
undefined    2150
bitter       1676
sour         1605
umami          58
Name: count, dtype: int64